# Lab 11 — False-Negative Forensics & Error Slicing

*AI Cybersecurity Masterclass · Lab 11 · AI60secLab*

**Research question:** Which attacks does the model miss, and what patterns distinguish those misses from correctly detected attacks?

Aggregate metrics tell us how many errors occurred. Forensics studies where those errors concentrate.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.forensics import (
    attach_prediction_context,
    error_slice_by_category,
    numeric_false_negative_contrast,
)

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

model = random_forest_pipeline(X_train)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
scores = model.predict_proba(X_test)[:, 1]

forensic = attach_prediction_context(
    test,
    y_test,
    predictions,
    scores,
)

display(forensic["error_type"].value_counts().to_frame())

In [ ]:
category_report = error_slice_by_category(
    forensic,
    "attack_cat",
)

display(category_report.round(4))

In [ ]:
false_negatives = forensic[
    forensic["error_type"] == "false_negative"
].copy()

columns_to_show = [
    c for c in [
        "attack_cat",
        "proto",
        "service",
        "state",
        "attack_score",
    ]
    if c in false_negatives.columns
]

display(
    false_negatives[columns_to_show]
    .sort_values("attack_score")
    .head(25)
)

In [ ]:
candidate_numeric = [
    "dur",
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "rate",
    "sload",
    "dload",
    "sinpkt",
    "dinpkt",
]

contrast = numeric_false_negative_contrast(
    forensic,
    candidate_numeric,
)

display(contrast.round(4))

## Build a research hypothesis

Choose one weak attack category.

Write:
- **Observation**
- **Hypothesis**
- **Experiment to test the hypothesis**
- **Alternative explanation**
- **Dataset limitation**